<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/04_workflows/01_sweeps_reports_automations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06. Sweeps / Reports / Automations

## この章のゴール
条件探索、共有、自動化の入口を確認します。

## 注目する W&B コード
```python
sweep_id = wandb.sweep(sweep_config, project=PROJECT)
wandb.agent(sweep_id, train, count=3)
```

In [ ]:
# 準備セル: そのまま実行します。
from pathlib import Path
import urllib.request

if not Path("course_utils.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py", "course_utils.py")

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb()

## 小さな Sweep を実行する
ここでは `lr` を 3 通り試し、各 run に `val/loss` を記録します。

In [ ]:
import wandb

sweep_config = {
    "method": "grid",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {"lr": {"values": [0.01, 0.03, 0.1]}},
}


def train():
    with wandb.init(**WANDB_INIT, tags=["course", "sweep"]) as run:
        lr = wandb.config.lr
        for step in range(5):
            loss = 1.0 / (step + 1) + lr
            run.log({"val/loss": loss, "step": step})

sweep_kwargs = {"project": PROJECT}
if ENTITY:
    sweep_kwargs["entity"] = ENTITY

sweep_id = wandb.sweep(sweep_config, **sweep_kwargs)
wandb.agent(sweep_id, train, count=3)
print("Sweep ID:", sweep_id)

## Reports で共有する内容
Report には、次の順番で結果を置くと読みやすくなります。

1. 目的: 何を比較したか
2. 条件: config のどこを変えたか
3. 結果: metric chart と Runs table
4. 定性評価: Table の代表例、失敗例
5. 次のアクション: 採用 run、追加実験、保留事項

## Automations の例
Automations は W&B App 上で設定します。権限、プラン、通知先の設定に依存するため、この講座では用途を確認します。

- 長時間学習が完了したら通知する
- run が failed / crashed になったら通知する
- model Artifact に alias が付いたら webhook を起動する

## W&B App で確認すること
- Sweeps に 3 runs が作成されている
- `val/loss` で条件を比較できる
- Reports と Automations が、結果共有と運用の入口になる